In [ ]:
import pandas as pd

# Carrega admissões e desligamentos gerais por município e ocupação
emprego_geral_cbo = pd.read_parquet(
    "../data/processed/trabalho/emprego_geral_municipio_cbo_2025.parquet"
)

# Verifica as dimensões e os nomes das colunas
print("Dimensões:", emprego_geral_cbo.shape)
print("Colunas:", emprego_geral_cbo.columns.tolist())

# Visualiza os primeiros registros
display(emprego_geral_cbo.head())

In [ ]:
# Carrega os dados de migrantes por município e ocupação
emprego_migrantes_cbo = pd.read_parquet(
    "../data/processed/trabalho/emprego_migrantes_municipio_cbo_2025.parquet"
)

# Padroniza os códigos CBO nas duas tabelas
emprego_geral_cbo["cbo2002ocupação"] = (
    emprego_geral_cbo["cbo2002ocupação"].astype("string").str.zfill(6)
)

emprego_migrantes_cbo["cbo2002ocupacao"] = (
    emprego_migrantes_cbo["cbo2002ocupacao"].astype("string").str.zfill(6)
)

# Diferencia os indicadores gerais e os de migrantes
emprego_geral_cbo = emprego_geral_cbo.rename(columns={
    "admissoes": "admissoes_gerais",
    "desligamentos": "desligamentos_gerais",
    "saldo": "saldo_geral"
})

emprego_migrantes_cbo = emprego_migrantes_cbo.rename(columns={
    "cbo2002ocupacao": "cbo",
    "admissoes": "admissoes_migrantes",
    "desligamentos": "desligamentos_migrantes",
    "saldo": "saldo_migrantes"
})

emprego_geral_cbo = emprego_geral_cbo.rename(
    columns={"cbo2002ocupação": "cbo"}
)

# Integra as duas bases por município e ocupação
comparacao_cbo_2025 = emprego_geral_cbo.merge(
    emprego_migrantes_cbo,
    on=["codigo_ibge", "cbo"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

# Verifica a cobertura e a integridade
print("Dimensões:", comparacao_cbo_2025.shape)
print("Códigos duplicados:",
      comparacao_cbo_2025.duplicated(["codigo_ibge", "cbo"]).sum())

display(comparacao_cbo_2025["_merge"].value_counts())

In [ ]:
# Seleciona combinações presentes nas duas bases
cbo_comparaveis = comparacao_cbo_2025[
    comparacao_cbo_2025["_merge"] == "both"
].copy()

# Verifica se as admissões de migrantes superam as gerais
inconsistencias = cbo_comparaveis[
    cbo_comparaveis["admissoes_migrantes"]
    > cbo_comparaveis["admissoes_gerais"]
]

# Resume a validação
print("Combinações comparadas:", len(cbo_comparaveis))
print("Combinações inconsistentes:", len(inconsistencias))

# Confere os totais de migrantes após a integração
print("Admissões migrantes:",
      cbo_comparaveis["admissoes_migrantes"].sum())

print("Desligamentos migrantes:",
      cbo_comparaveis["desligamentos_migrantes"].sum())

# Mostra eventuais inconsistências
display(
    inconsistencias[
        ["codigo_ibge", "cbo",
         "admissoes_migrantes", "admissoes_gerais"]
    ].head(20)
)

In [ ]:
# Agrupa as admissões gerais e de migrantes por ocupação CBO
ranking_comparativo_cbo = (
    comparacao_cbo_2025
    .groupby("cbo", dropna=False)
    .agg(
        admissoes_gerais=("admissoes_gerais", "sum"),
        admissoes_migrantes=("admissoes_migrantes", "sum")
    )
    .reset_index()
)

# Calcula a participação de migrantes nas admissões de cada ocupação
ranking_comparativo_cbo["participacao_migrante_pct"] = (
    ranking_comparativo_cbo["admissoes_migrantes"]
    / ranking_comparativo_cbo["admissoes_gerais"] * 100
).round(2)

# Carrega o dicionário oficial de ocupações CBO
ocupacoes_cbo = pd.read_csv(
    "../data/raw/trabalho/cbo/cbo2002-ocupacao.csv",
    sep=";",
    encoding="latin-1",
    dtype={"CODIGO": "string"}
)

# Padroniza os códigos e associa os títulos das ocupações
ocupacoes_cbo["CODIGO"] = ocupacoes_cbo["CODIGO"].str.zfill(6)

ranking_comparativo_cbo = ranking_comparativo_cbo.merge(
    ocupacoes_cbo[["CODIGO", "TITULO"]],
    left_on="cbo",
    right_on="CODIGO",
    how="left",
    validate="many_to_one"
)

# Exibe as 20 ocupações com mais admissões de migrantes
display(
    ranking_comparativo_cbo.nlargest(20, "admissoes_migrantes")[
        ["cbo", "TITULO", "admissoes_gerais",
         "admissoes_migrantes", "participacao_migrante_pct"]
    ]
)

In [ ]:
# Calcula a participação migrante no conjunto das admissões
participacao_referencia = (
    ranking_comparativo_cbo["admissoes_migrantes"].sum()
    / ranking_comparativo_cbo["admissoes_gerais"].sum() * 100
)

# Mede a representação migrante relativa à referência geral
ranking_comparativo_cbo["indice_representacao"] = (
    ranking_comparativo_cbo["participacao_migrante_pct"]
    / participacao_referencia
)

# Exibe as ocupações com mais admissões migrantes
print(f"Participação de referência: {participacao_referencia:.2f}%")

display(
    ranking_comparativo_cbo.nlargest(20, "admissoes_migrantes")[
        ["TITULO", "admissoes_gerais", "admissoes_migrantes",
         "participacao_migrante_pct", "indice_representacao"]
    ]
)

In [ ]:
# Seleciona ocupações com volume mínimo de admissões
oportunidades_municipais_cbo = comparacao_cbo_2025[
    (comparacao_cbo_2025["admissoes_gerais"] >= 100) &
    (comparacao_cbo_2025["admissoes_migrantes"] >= 30)
].copy()

# Calcula a participação migrante em cada município e ocupação
oportunidades_municipais_cbo["participacao_migrante_pct"] = (
    oportunidades_municipais_cbo["admissoes_migrantes"]
    / oportunidades_municipais_cbo["admissoes_gerais"] * 100
).round(2)

# Adiciona o nome das ocupações
oportunidades_municipais_cbo = oportunidades_municipais_cbo.merge(
    ocupacoes_cbo[["CODIGO", "TITULO"]],
    left_on="cbo",
    right_on="CODIGO",
    how="left",
    validate="many_to_one"
)

# Exibe as maiores participações migrantes
display(
    oportunidades_municipais_cbo.nlargest(
        20, "participacao_migrante_pct"
    )[
        ["codigo_ibge", "TITULO", "admissoes_gerais",
         "admissoes_migrantes", "participacao_migrante_pct"]
    ]
)

In [ ]:
# Carrega os nomes dos municípios já padronizados
municipios = pd.read_parquet(
    "../data/processed/indicadores_completos_municipais.parquet"
)[["codigo_ibge", "municipio", "uf"]].drop_duplicates("codigo_ibge")

In [ ]:
# Acrescenta município e UF às oportunidades por ocupação

oportunidades_municipais_cbo = oportunidades_municipais_cbo.merge(
    municipios[["codigo_ibge", "municipio"]],
    on="codigo_ibge",
    how="left",
    validate="many_to_one"
)

In [ ]:
# Exibe o Top 20 com município e UF
display(
    oportunidades_municipais_cbo.nlargest(
        20, "participacao_migrante_pct"
    )[
        ["municipio", "uf", "TITULO",
         "admissoes_gerais", "admissoes_migrantes",
         "participacao_migrante_pct"]
    ]
)

In [ ]:
# Carrega a oferta de cursos técnicos por município
oferta_cursos = pd.read_parquet(
    "../data/processed/educacao/educacao_municipio_curso_2025.parquet"
)

# Verifica a estrutura e alguns exemplos
print("Dimensões:", oferta_cursos.shape)
print("Colunas:", oferta_cursos.columns.tolist())

display(oferta_cursos.head())

In [ ]:
# Resume a oferta de cursos por área de formação
resumo_areas = (
    oferta_cursos
    .groupby("NO_AREA_CURSO_PROFISSIONAL")
    .agg(
        cursos_distintos=("CO_CURSO_EDUC_PROFISSIONAL", "nunique"),
        municipios=("codigo_ibge", "nunique"),
        matriculas=("matriculas", "sum")
    )
    .sort_values("matriculas", ascending=False)
)

display(resumo_areas)